# Drive → SRT (ruso) con faster-whisper large-v3

Notebook autocontenido. Levanta videos desde tu Drive (carpeta **Host Videos**), los transcribe a SRT en ruso con `large-v3` y guarda los `.srt` en tu Drive (carpeta **Subs_RU**).

**Son 2 celdas:** la primera prepara todo y monta Drive; la segunda transcribe y guarda. Suena un ruidito cuando termina.

**Antes de correr:** `Runtime → Change runtime type → T4 GPU` (o cualquier GPU con soporte de `float16`). En CPU corre pero muy lento y con `int8` (no `float16`).

**Parámetros de Whisper** (equivalentes a tu llamada de PowerShell, salvo donde se aclara):
- `model=large-v3`, `language=ru`, `task=transcribe`
- `compute_type=float16` (GPU), `beam_size=10`, `temperature=0`
- `condition_on_previous_text=False`
- `compression_ratio_threshold=2.0`
- `vad_filter=True`, `vad_parameters={"threshold": 0.4}`
- `initial_prompt` con términos de Romanov (пн-переход, качер, варикап…)
- Post-proceso para emular los flags del CLI `faster-whisper-xxl` que la API Python no expone: `--sentence` (cortes por oración usando `word_timestamps`), `--max_line_width 200` y `--max_line_count 1` (una sola línea por cue, hasta 200 caracteres; si una oración se pasa, se parte en sub-cues sin cortar palabras).


## 1) Setup + montar Drive

Corré esta celda. Instala todo, monta tu Drive y lista los archivos que va a transcribir.

- **Entrada:** `MyDrive/Host Videos` — acepta `.mp4 .mkv .webm .mov .avi .m4v .m4a .mp3 .wav .ogg .opus .aac .flac`.
- **Salida:** `MyDrive/Subs_RU` — se crea si no existe. Si un SRT ya está ahí, en el paso 2 se saltea (re-runs seguros).

Si tus carpetas se llaman distinto, editá `INPUT_DIR` / `OUTPUT_DIR` abajo.


In [ ]:
!pip install -q faster-whisper
!apt-get -qq install -y ffmpeg > /dev/null

import os, time, re
from pathlib import Path
from google.colab import drive
import torch

# --- Mount Drive ---
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")
print("Drive montado.")

# --- Carpetas (editar si tus nombres son distintos) ---
INPUT_DIR  = Path("/content/drive/MyDrive/Host Videos")
OUTPUT_DIR = Path("/content/drive/MyDrive/Subs_RU")

assert INPUT_DIR.exists(), f"No existe la carpeta de entrada: {INPUT_DIR}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_EXTS = {".mp4", ".mkv", ".webm", ".mov", ".avi", ".m4v",
              ".m4a", ".mp3", ".wav", ".ogg", ".opus", ".aac", ".flac"}
inputs = sorted(p for p in INPUT_DIR.iterdir()
                if p.is_file() and p.suffix.lower() in VIDEO_EXTS)
assert inputs, f"No hay archivos de video/audio en {INPUT_DIR}"

print(f"\n{len(inputs)} archivo(s) en '{INPUT_DIR.name}':")
for p in inputs:
    srt = OUTPUT_DIR / f"{p.stem}-RU.srt"
    status = "✓ ya existe" if srt.exists() else "→ pendiente"
    print(f"  [{status}] {p.name}")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"
print(f"\nDevice: {DEVICE}  |  compute_type: {COMPUTE_TYPE}")
if DEVICE == "cuda":
    print(torch.cuda.get_device_name(0))
else:
    print("[WARN] sin GPU: 'float16' no aplica, cae a 'int8'. Será lento.")


## 2) Transcribir todo → SRT en Drive

Una sola celda hace el resto:

1. Carga el modelo `large-v3` (una sola vez; si re-corrés la celda, lo reusa).
2. Transcribe cada archivo con los parámetros descriptos arriba.
3. Aplica post-proceso `--sentence` + `--max_line_width 200` + `--max_line_count 1`: agrupa palabras (con sus timestamps) en oraciones cortando en `. ! ? …`, y si una oración se pasa de 200 chars la parte en sub-cues sin cortar palabras (cada cue, una sola línea).
4. Guarda el SRT en `MyDrive/Subs_RU/<stem>-RU.srt`. Si ya existe, lo saltea.


In [ ]:
from faster_whisper import WhisperModel
from IPython.display import Audio, display
import numpy as np

INITIAL_PROMPT = (
    "Лекция Александра Романова о бестопливных генераторах. Термины: ПН-переход, разрядник, качер, варикап, тиристор, лавинный диод, туннельный диод, Тесла, фузьки, бифилярная катушка."
)

SENTENCE_END = (".", "!", "?", "…")

def fmt_ts(t):
    h = int(t // 3600)
    m = int((t % 3600) // 60)
    s = int(t % 60)
    ms = int(round((t - int(t)) * 1000))
    if ms == 1000:
        ms = 0
        s += 1
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

def _split_long(words, max_width):
    """Parte una oración larga en sub-cues <= max_width sin cortar palabras."""
    chunk = []
    chunk_len = 0
    for w in words:
        piece = w.word
        if chunk and chunk_len + len(piece) > max_width:
            text = "".join(x.word for x in chunk).strip()
            if text:
                yield {"start": chunk[0].start, "end": chunk[-1].end, "text": text}
            chunk = [w]
            chunk_len = len(piece)
        else:
            chunk.append(w)
            chunk_len += len(piece)
    if chunk:
        text = "".join(x.word for x in chunk).strip()
        if text:
            yield {"start": chunk[0].start, "end": chunk[-1].end, "text": text}

def collect_cues(segments, max_width=200):
    """
    Itera segments con word_timestamps=True, agrupa words en oraciones
    (cortando en . ! ? …), y emite cues de una sola línea <= max_width.
    """
    buf = []
    for seg in segments:
        for w in (seg.words or []):
            buf.append(w)
            tail = w.word.strip()
            if tail and tail[-1] in SENTENCE_END:
                text = "".join(x.word for x in buf).strip()
                if len(text) <= max_width:
                    if text:
                        yield {"start": buf[0].start, "end": buf[-1].end, "text": text}
                else:
                    yield from _split_long(buf, max_width)
                buf = []
    if buf:
        text = "".join(x.word for x in buf).strip()
        if len(text) <= max_width:
            if text:
                yield {"start": buf[0].start, "end": buf[-1].end, "text": text}
        else:
            yield from _split_long(buf, max_width)

# --- Cargar modelo una sola vez ---
if "model" not in globals():
    print("Cargando modelo large-v3 (la primera vez tarda)...")
    model = WhisperModel("large-v3", device=DEVICE, compute_type=COMPUTE_TYPE)
    print("Modelo listo.")

t_global = time.time()
done = skipped = failed = 0
for i, vid in enumerate(inputs, 1):
    srt = OUTPUT_DIR / f"{vid.stem}-RU.srt"
    if srt.exists():
        print(f"\n[{i}/{len(inputs)}] SALTADO (ya existe en Drive): {srt.name}")
        skipped += 1
        continue
    print(f"\n[{i}/{len(inputs)}] Procesando: {vid.name}")
    t0 = time.time()
    try:
        segments, info = model.transcribe(
            str(vid),
            language="ru",
            task="transcribe",
            beam_size=10,
            temperature=0,
            condition_on_previous_text=False,
            compression_ratio_threshold=2.0,
            vad_filter=True,
            vad_parameters={"threshold": 0.4},
            initial_prompt=INITIAL_PROMPT,
            word_timestamps=True,
        )
        print(f"   duración audio: {info.duration:.1f}s, lang detectado: {info.language}")
        cues = list(collect_cues(segments, max_width=200))
        with open(srt, "w", encoding="utf-8") as f:
            for n, c in enumerate(cues, 1):
                f.write(f"{n}\n{fmt_ts(c['start'])} --> {fmt_ts(c['end'])}\n{c['text']}\n\n")
        print(f"[{i}/{len(inputs)}] Listo ({len(cues)} cues, {time.time()-t0:.1f}s) → {srt.name}")
        done += 1
    except Exception as ex:
        print(f"[{i}/{len(inputs)}] FALLÓ: {ex}")
        failed += 1

print(f"\n=== Resumen ===")
print(f"  procesados: {done}")
print(f"  saltados:   {skipped}")
print(f"  fallidos:   {failed}")
print(f"  tiempo total: {(time.time()-t_global)/60:.1f} min")
print(f"\nSRT en: {OUTPUT_DIR}")

# Ruidito final
sr = 22050
out = np.array([], dtype=np.float32)
for f in [392, 523, 659, 784, 1047]:
    t = np.linspace(0, 0.18, int(sr*0.18), endpoint=False)
    env = np.exp(-3*t)
    out = np.concatenate([out, (0.3*env*np.sin(2*np.pi*f*t)).astype(np.float32)])
display(Audio(out, rate=sr, autoplay=True))
